This notebook implements the PostgreSQL database workflow for the fintech review analytics project.

The notebook covers:
- database connection,
- relational schema usage,
- data loading,
- SQL validation queries,
- and analytical SQL queries for business insights.

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

In [2]:
load_dotenv("../.env")

DB_NAME = os.getenv("DB_NAME")

DB_USER = os.getenv("DB_USERNAME")

DB_PASSWORD = os.getenv("DB_PASSWORD")

DB_HOST = os.getenv("DB_HOST")

DB_PORT = os.getenv("DB_PORT")

print("Environment variables loaded successfully.")

Environment variables loaded successfully.


In [3]:
print(DB_NAME)

print(DB_USER)

print(DB_HOST)

print(DB_PORT)

fintech_reviews_db
postgres
localhost
5432


In [4]:
DATABASE_URL = (
    f"postgresql://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)

print("Database connection successful.")

Database connection successful.


In [5]:
df = pd.read_csv(
    "../data/raw/bank_reviews_cleaned.csv"
)

df.head()

,review_id,review,rating,date,bank,source
0,2db3aee9378ee99f0b1a0d06c89472e3,🤙🏼🤙🏼,5,2026-05-16,CBE,Google Play
1,afb48085f9b66590cd52d9ddd99298e9,worst,1,2026-05-16,CBE,Google Play
2,7e93f62c010d74d6d47991cf82c0b683,this app very full,5,2026-05-16,CBE,Google Play
3,4d0e52a77bec925497723ba43927be53,good apps,4,2026-05-16,CBE,Google Play
4,f13a98bc99db9a556697ca54fec6667e,ok,5,2026-05-16,CBE,Google Play


### Database Schema Overview

The PostgreSQL database was normalized into the following tables:

1. `banks`
2. `reviews`
3. `sentiments`

The schema design improves:
- scalability,
- maintainability,
- and relational consistency.

In [6]:
banks_df = pd.DataFrame({
    "bank_name": df["bank"].unique()
})

banks_df

,bank_name
0,CBE
1,BOA
2,Dashen


In [7]:
banks_df.to_sql(
    "banks",
    engine,
    if_exists="append",
    index=False
)

print("Banks inserted successfully.")

Banks inserted successfully.


In [8]:
bank_lookup = pd.read_sql(
    "SELECT * FROM banks",
    engine
)

bank_lookup

,bank_id,bank_name
0,7,CBE
1,8,BOA
2,9,Dashen


In [9]:
bank_map = dict(
    zip(
        bank_lookup["bank_name"],
        bank_lookup["bank_id"]
    )
)

df["bank_id"] = df["bank"].map(bank_map)

df.head()

,review_id,review,rating,date,bank,source,bank_id
0,2db3aee9378ee99f0b1a0d06c89472e3,🤙🏼🤙🏼,5,2026-05-16,CBE,Google Play,7
1,afb48085f9b66590cd52d9ddd99298e9,worst,1,2026-05-16,CBE,Google Play,7
2,7e93f62c010d74d6d47991cf82c0b683,this app very full,5,2026-05-16,CBE,Google Play,7
3,4d0e52a77bec925497723ba43927be53,good apps,4,2026-05-16,CBE,Google Play,7
4,f13a98bc99db9a556697ca54fec6667e,ok,5,2026-05-16,CBE,Google Play,7


In [12]:
reviews_df = df[[
    "review_id",
    "bank_id",
    "review",
    "rating",
    "date",
    "source"
]].copy()

reviews_df.columns = [
    "review_id",
    "bank_id",
    "review",
    "rating",
    "review_date",
    "source"
]

reviews_df.head()

,review_id,bank_id,review,rating,review_date,source
0,2db3aee9378ee99f0b1a0d06c89472e3,7,🤙🏼🤙🏼,5,2026-05-16,Google Play
1,afb48085f9b66590cd52d9ddd99298e9,7,worst,1,2026-05-16,Google Play
2,7e93f62c010d74d6d47991cf82c0b683,7,this app very full,5,2026-05-16,Google Play
3,4d0e52a77bec925497723ba43927be53,7,good apps,4,2026-05-16,Google Play
4,f13a98bc99db9a556697ca54fec6667e,7,ok,5,2026-05-16,Google Play


In [13]:
reviews_df.to_sql(
    "reviews",
    engine,
    if_exists="append",
    index=False
)

print("Reviews inserted successfully.")

Reviews inserted successfully.


### Validation Queries

The following queries verify:
- successful data insertion,
- relational consistency,
- and review distribution across banks.

In [14]:
query = """
SELECT COUNT(*) AS total_reviews
FROM reviews;
"""

pd.read_sql(query, engine)

,total_reviews
0,3000


In [15]:
query = """
SELECT
    b.bank_name,
    COUNT(r.review_id) AS total_reviews
FROM reviews r
JOIN banks b
ON r.bank_id = b.bank_id
GROUP BY b.bank_name;
"""

pd.read_sql(query, engine)

,bank_name,total_reviews
0,BOA,1000
1,CBE,1000
2,Dashen,1000


In [16]:
query = """
SELECT
    b.bank_name,
    ROUND(AVG(r.rating), 2) AS average_rating
FROM reviews r
JOIN banks b
ON r.bank_id = b.bank_id
GROUP BY b.bank_name;
"""

pd.read_sql(query, engine)

,bank_name,average_rating
0,BOA,3.23
1,CBE,4.07
2,Dashen,4.16


### Early Database Insights

The SQL validation queries confirm that:
- review records were successfully inserted,
- relational joins are functioning correctly,
- and customer review distributions vary across banks.

The average ratings also provide an early indication of customer satisfaction differences between banking applications.

In [18]:
query = """
SELECT
    b.bank_name,
    MIN(r.rating) AS minimum_rating,
    MAX(r.rating) AS maximum_rating
FROM reviews r
JOIN banks b
ON r.bank_id = b.bank_id
GROUP BY b.bank_name;
"""

pd.read_sql(query, engine)

,bank_name,minimum_rating,maximum_rating
0,BOA,1,5
1,CBE,1,5
2,Dashen,1,5


### Conclusion

This notebook successfully implemented the PostgreSQL database engineering workflow for the fintech review analytics project.

The workflow included:
- relational schema implementation,
- PostgreSQL integration,
- structured data loading,
- SQL validation queries,
- and analytical querying.

The database architecture now supports scalable downstream analytics and business intelligence workflows.